# CIFAR-10 Image Classification with Convolutional Neural Networks (CNNs)

**CSCI 394 - Spring 2026 Tutorial**

In the previous tutorial, we saw that a fully connected network achieves only ~55% accuracy on CIFAR-10. In this tutorial, we build a **Convolutional Neural Network (CNN)** that dramatically improves performance by exploiting the spatial structure of images.

## What you will learn

1. How convolution operations work and why they are ideal for images
2. How to build a CNN with convolutional, pooling, and batch normalization layers
3. How data augmentation improves generalization
4. How to compare CNN vs. fully connected network performance
5. How to visualize learned convolutional filters

## Background: Convolutional Neural Networks

### The Key Insight

Images have **local spatial structure**: nearby pixels are more related than distant ones. A CNN exploits this by:

1. **Local connectivity**: Each neuron connects only to a small region of the input (the "receptive field")
2. **Parameter sharing**: The same filter (set of weights) is applied across the entire image
3. **Hierarchical features**: Early layers detect edges, middle layers detect textures/parts, and deep layers detect objects

### Core CNN Operations

| Operation | What it does | Why it matters |
|-----------|-------------|----------------|
| **Conv2d** | Slides a small filter across the image, computing dot products | Detects local patterns (edges, textures) with shared weights |
| **MaxPool2d** | Takes the maximum value in each patch | Reduces spatial dimensions, adds translation invariance |
| **BatchNorm** | Normalizes activations within each batch | Stabilizes training, allows higher learning rates |
| **ReLU** | `max(0, x)` | Non-linearity between layers |
| **Dropout** | Randomly zeros neurons during training | Prevents overfitting |

### How Convolution Works

A 3x3 convolution filter slides across the input image:

```
Input image (e.g., 32x32x3)      Filter (3x3x3)       Output feature map
  +-----------+                    +---+
  | . . . . . |                    |w w|                 +---------+
  | . [x x x] |  *  (element-     |w w|  =  sum  ->     | . . . . |
  | . [x x x] |     wise mult     |w w|                 | . [y] . |
  | . [x x x] |     + sum)        +---+                 | . . . . |
  | . . . . . |                                          +---------+
  +-----------+
```

The filter is slid to every position, producing one output value per position. Multiple filters produce multiple **feature maps** (channels).

## Step 0: Setup (Google Colab)

Enable GPU: `Runtime > Change runtime type > GPU`.

In [ ]:
# Install dependencies (only needed if not on Colab)
# !pip install torch torchvision matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import time

print(f"PyTorch version: {torch.__version__}")

# Select device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU (training will be slower)")

print(f"Device: {device}")

## Step 1: Load CIFAR-10 with Data Augmentation

### Data Augmentation

Data augmentation artificially expands the training set by applying random transformations to images. This acts as a regularizer and helps the model learn invariances.

Common augmentations for image classification:

- **Random horizontal flip**: Mirror the image left-to-right (a cat is still a cat when flipped)
- **Random crop with padding**: Shift the image slightly in any direction (objects can appear anywhere)

**Important**: Augmentation is applied only to training data, not to test data!

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
EPOCHS = 25
LEARNING_RATE = 1e-3
SEED = 42

torch.manual_seed(SEED)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Training transform: augmentation + normalization
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),     # Randomly shift the image
    transforms.RandomHorizontalFlip(),         # 50% chance of horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

# Test transform: only normalization (no augmentation!)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Image shape:      {train_dataset[0][0].shape}  (C x H x W)")
print(f"Batches per epoch: {len(train_loader)}")

### Visualize the effect of data augmentation

The same image looks different each time augmentation is applied.

In [ ]:
def unnormalize(img):
    """Reverse normalization for display."""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    return (img * std + mean).clamp(0, 1)

# Get a raw image (without augmentation) for reference
raw_dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=transforms.ToTensor())
original_img, label = raw_dataset[0]

# Show the original and several augmented versions
fig, axes = plt.subplots(1, 7, figsize=(16, 3))
axes[0].imshow(original_img.permute(1, 2, 0).numpy())
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

# The raw PIL image for augmentation
raw_pil_dataset = datasets.CIFAR10(root='./data', train=True, download=False)
pil_img = raw_pil_dataset[0][0]

for i in range(1, 7):
    augmented = train_transform(pil_img)
    axes[i].imshow(unnormalize(augmented).permute(1, 2, 0).numpy())
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.suptitle(f'Data Augmentation Examples (class: {CLASS_NAMES[label]})', fontsize=14)
plt.tight_layout()
plt.show()

## Step 2: Build the CNN

Our CNN architecture has three convolutional blocks followed by a classifier:

```
Input: 3 x 32 x 32

Block 1: Conv(3->32, 3x3) -> BatchNorm -> ReLU -> Conv(32->32, 3x3) -> BatchNorm -> ReLU -> MaxPool(2x2)
  Output: 32 x 16 x 16

Block 2: Conv(32->64, 3x3) -> BatchNorm -> ReLU -> Conv(64->64, 3x3) -> BatchNorm -> ReLU -> MaxPool(2x2)
  Output: 64 x 8 x 8  (with padding)

Block 3: Conv(64->128, 3x3) -> BatchNorm -> ReLU -> Conv(128->128, 3x3) -> BatchNorm -> ReLU -> MaxPool(2x2)
  Output: 128 x 4 x 4  (with padding)

Classifier: Flatten -> Dropout(0.5) -> Linear(128*4*4, 256) -> ReLU -> Dropout(0.3) -> Linear(256, 10)
```

### Key design choices:

- **Two conv layers per block**: Increases the receptive field without pooling too aggressively
- **Batch normalization**: Normalizes each channel's activations, accelerating convergence
- **Increasing channels** (32 -> 64 -> 128): As spatial dimensions shrink, we increase the number of feature maps to preserve information capacity
- **`padding=1`** on 3x3 convolutions: Preserves the spatial dimensions (same padding)

In [ ]:
class CIFAR10CNN(nn.Module):
    """CNN for CIFAR-10 classification."""
    
    def __init__(self):
        super().__init__()
        
        # Convolutional feature extractor
        self.features = nn.Sequential(
            # Block 1: 3x32x32 -> 32x16x16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 2: 32x16x16 -> 64x8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Block 3: 64x8x8 -> 128x4x4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CIFAR10CNN().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model architecture:\n{model}")
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

### Understanding feature map dimensions

Let's trace how the spatial dimensions change through the network.

In [ ]:
# Trace dimensions through the network
print("Feature map dimensions through the network:")
print("-" * 55)
x = torch.randn(1, 3, 32, 32).to(device)
print(f"{'Layer':40s} {'Output Shape':>12}")
print(f"{'Input':40s} {str(list(x.shape)):>12}")

for i, layer in enumerate(model.features):
    x = layer(x)
    name = f"{layer.__class__.__name__}"
    if hasattr(layer, 'in_channels'):
        name += f"({layer.in_channels}->{layer.out_channels})"
    elif hasattr(layer, 'num_features'):
        name += f"({layer.num_features})"
    print(f"{name:40s} {str(list(x.shape)):>12}")

for i, layer in enumerate(model.classifier):
    x = layer(x)
    name = f"{layer.__class__.__name__}"
    if hasattr(layer, 'in_features'):
        name += f"({layer.in_features}->{layer.out_features})"
    print(f"{name:40s} {str(list(x.shape)):>12}")

## Step 3: Loss, Optimizer, and Learning Rate Scheduler

### Learning Rate Scheduling

A learning rate scheduler adjusts the learning rate during training. We use **ReduceLROnPlateau**, which reduces the learning rate when the validation loss stops improving. This allows the model to take large steps early in training and fine-tune with smaller steps later.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)")
print(f"  -> Halves the learning rate if test loss doesn't improve for 3 consecutive epochs")

## Step 4: Training Loop

In [ ]:
def evaluate(model, loader, loss_fn, device):
    """Evaluate the model on a dataset."""
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = loss_fn(logits, labels)
            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_seen += labels.size(0)
    
    return total_loss / total_seen, total_correct / total_seen

In [ ]:
# Training loop
train_losses = []
test_losses = []
train_accs = []
test_accs = []
learning_rates = []

print(f"Training CNN on {device} for {EPOCHS} epochs...")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Loss':>9} | {'Test Acc':>8} | {'LR':>10} | {'Time':>6}")
print("-" * 78)

total_start = time.time()
best_test_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_seen = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        logits = model(images)
        loss = loss_fn(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
        running_correct += (logits.argmax(dim=1) == labels).sum().item()
        running_seen += labels.size(0)
    
    train_loss = running_loss / running_seen
    train_acc = running_correct / running_seen
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    
    # Step the scheduler based on test loss
    scheduler.step(test_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    learning_rates.append(current_lr)
    
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_marker = " *"
    else:
        best_marker = ""
    
    elapsed = time.time() - epoch_start
    print(f"{epoch:5d} | {train_loss:10.4f} | {train_acc:8.2%} | {test_loss:9.4f} | {test_acc:7.2%} | {current_lr:10.6f} | {elapsed:5.1f}s{best_marker}")

total_time = time.time() - total_start
print(f"\nTotal training time: {total_time:.1f}s")
print(f"Best test accuracy:  {best_test_acc:.2%}")
print(f"Final test accuracy: {test_accs[-1]:.2%}")

## Step 5: Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, EPOCHS + 1)

# Loss
axes[0].plot(epochs_range, train_losses, 'b-o', label='Train', markersize=3)
axes[0].plot(epochs_range, test_losses, 'r-o', label='Test', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [a * 100 for a in train_accs], 'b-o', label='Train', markersize=3)
axes[1].plot(epochs_range, [a * 100 for a in test_accs], 'r-o', label='Test', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(epochs_range, learning_rates, 'g-o', markersize=3)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.suptitle('CNN Training on CIFAR-10', fontsize=14)
plt.tight_layout()
plt.show()

## Step 6: Per-Class Analysis

In [ ]:
# Get all predictions
model.eval()
all_preds = []
all_labels = []
all_images = []

with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        preds = logits.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
        all_images.append(images)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
all_images = torch.cat(all_images)

# Per-class accuracy
print("Per-class accuracy:")
print("-" * 35)
class_accs = []
for i in range(10):
    mask = all_labels == i
    class_acc = (all_preds[mask] == all_labels[mask]).float().mean().item()
    class_accs.append(class_acc)
    print(f"  {CLASS_NAMES[i]:12s}: {class_acc:.2%}")

plt.figure(figsize=(10, 5))
colors = ['green' if a > 0.8 else 'orange' if a > 0.6 else 'red' for a in class_accs]
plt.bar(range(10), [a * 100 for a in class_accs], color=colors, edgecolor='black')
plt.xticks(range(10), CLASS_NAMES, rotation=45, ha='right')
plt.ylabel('Accuracy (%)')
plt.title('CNN Per-Class Test Accuracy')
plt.axhline(y=np.mean(class_accs) * 100, color='blue', linestyle='--', 
            label=f'Mean: {np.mean(class_accs):.1%}')
plt.legend()
plt.tight_layout()
plt.show()

## Step 7: Confusion Matrix

In [ ]:
confusion = torch.zeros(10, 10, dtype=torch.int64)
for pred, true in zip(all_preds, all_labels):
    confusion[true, pred] += 1

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(confusion, cmap='Blues')

for i in range(10):
    for j in range(10):
        color = 'white' if confusion[i, j] > confusion.max() / 2 else 'black'
        ax.text(j, i, f'{confusion[i, j]}', ha='center', va='center', color=color, fontsize=8)

ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('CNN Confusion Matrix', fontsize=14)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels(CLASS_NAMES)
plt.colorbar(im)
plt.tight_layout()
plt.show()

## Step 8: Visualize Learned Filters

One advantage of CNNs is that we can inspect what the convolutional filters have learned. The first layer's filters are directly interpretable because they operate on RGB input.

In [ ]:
# Visualize the first convolutional layer's filters
first_conv = model.features[0]
filters = first_conv.weight.data.cpu()

# Normalize filters for display
filters_min = filters.min()
filters_max = filters.max()
filters_norm = (filters - filters_min) / (filters_max - filters_min)

n_filters = filters.shape[0]
cols = 8
rows = (n_filters + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 1.8))
for i in range(rows * cols):
    ax = axes[i // cols, i % cols]
    if i < n_filters:
        # Show as RGB image (3x3x3 filter)
        ax.imshow(filters_norm[i].permute(1, 2, 0).numpy())
        ax.set_title(f'Filter {i}', fontsize=8)
    ax.axis('off')

plt.suptitle('First Conv Layer Learned Filters (3x3 RGB)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 9: Visualize Feature Maps

Let's see what the network "sees" at different layers when processing an image.

In [ ]:
# Pick a test image
test_img, test_label = test_dataset[7]
print(f"Input image class: {CLASS_NAMES[test_label]}")

# Show the input image
plt.figure(figsize=(3, 3))
plt.imshow(unnormalize(test_img).permute(1, 2, 0).numpy())
plt.title(f'Input: {CLASS_NAMES[test_label]}')
plt.axis('off')
plt.show()

# Pass through each layer and capture feature maps at key points
model.eval()
x = test_img.unsqueeze(0).to(device)  # Add batch dimension

feature_maps = []
layer_names = []

for i, layer in enumerate(model.features):
    x = layer(x)
    if isinstance(layer, nn.ReLU):
        feature_maps.append(x.detach().cpu().squeeze(0))
        # Determine which block we're in
        layer_names.append(f'After ReLU (block {len(feature_maps)//2 + 1}, ch={x.shape[1]})')

# Show feature maps from blocks 1, 2, and 3 (after the first ReLU in each block)
for idx in [0, 2, 4]:  # First ReLU of each block
    if idx < len(feature_maps):
        fmaps = feature_maps[idx]
        n_show = min(16, fmaps.shape[0])  # Show up to 16 channels
        
        fig, axes = plt.subplots(2, 8, figsize=(16, 4))
        for i in range(n_show):
            ax = axes[i // 8, i % 8]
            ax.imshow(fmaps[i].numpy(), cmap='viridis')
            ax.set_title(f'Ch {i}', fontsize=8)
            ax.axis('off')
        plt.suptitle(f'{layer_names[idx]} — Feature Maps ({fmaps.shape[1]}x{fmaps.shape[2]})', fontsize=12)
        plt.tight_layout()
        plt.show()

## Step 10: Compare CNN vs. Fully Connected Network

Let's summarize the comparison between the FC network (previous tutorial) and this CNN.

In [ ]:
# Summary comparison table
print("=" * 60)
print("COMPARISON: Fully Connected vs. CNN on CIFAR-10")
print("=" * 60)
print(f"{'Metric':<25} {'FC Network':>15} {'CNN':>15}")
print("-" * 60)
print(f"{'Total parameters':<25} {'~1.7M':>15} {f'{total_params:,}':>15}")
print(f"{'Expected test accuracy':<25} {'~50-55%':>15} {f'{best_test_acc:.1%}':>15}")
print(f"{'Data augmentation':<25} {'No':>15} {'Yes':>15}")
print(f"{'LR scheduling':<25} {'No':>15} {'Yes':>15}")
print(f"{'Batch normalization':<25} {'No':>15} {'Yes':>15}")
print(f"{'Spatial awareness':<25} {'No':>15} {'Yes':>15}")
print(f"{'Translation invariance':<25} {'No':>15} {'Yes':>15}")
print("=" * 60)
print("\nThe CNN achieves significantly higher accuracy with comparable")
print("or fewer parameters, thanks to its inductive bias for spatial data.")

## Summary

### Key takeaways

| Aspect | Details |
|--------|--------|
| **Dataset** | CIFAR-10: 50k train / 10k test, 32x32 RGB images, 10 classes |
| **Model** | 6-layer CNN (3 blocks of 2 convolutions + pooling) |
| **Techniques** | Batch normalization, dropout, data augmentation, LR scheduling |
| **Expected accuracy** | ~80-85% on test set |

### Why CNNs work better than FC networks for images

1. **Local feature detection**: Convolutions detect patterns regardless of position
2. **Parameter efficiency**: A 3x3 filter has only 27 parameters (vs. thousands in FC layers)
3. **Hierarchical representation**: Edges -> Textures -> Parts -> Objects
4. **Translation invariance**: Pooling makes the network robust to small shifts

### Going further

To push accuracy even higher on CIFAR-10, you could:

1. **Use deeper architectures**: ResNet, VGG, DenseNet (can reach 93%+)
2. **More data augmentation**: Cutout, MixUp, AutoAugment
3. **Cosine annealing scheduler**: Often works better than ReduceLROnPlateau
4. **Weight decay**: L2 regularization via `optimizer = Adam(..., weight_decay=1e-4)`

### Exercises

1. **Remove batch normalization**: Comment out all `BatchNorm2d` layers. How does training stability change?
2. **Remove data augmentation**: Use `test_transform` for training too. How much does accuracy drop?
3. **Increase depth**: Add a fourth convolutional block with 256 channels. Does it help?
4. **Try different kernel sizes**: Replace 3x3 kernels with 5x5. What is the trade-off?
5. **Global Average Pooling**: Replace `Flatten + Linear(128*4*4, 256)` with `nn.AdaptiveAvgPool2d(1)` followed by `Linear(128, 10)`. This eliminates many parameters.